# 🏗️ Databricks Medallion Architecture (Bronze -> Silver -> Gold)
## Real-Time Customer Churn Prediction & Intelligent Retention Platform

This notebook demonstrates the end-to-end **Databricks Delta Lake Medallion Pipeline**:
1. **Bronze Tier**: Raw ingestion of IBM Telco customer data and simulated streaming events with metadata lineage (`_ingested_at`, `_batch_id`).
2. **Silver Tier**: Data cleaning, constraint validation, type casting, missing value handling, and deduplication.
3. **Gold Tier**: High-value aggregated customer feature store (RFM metrics, service bundle scores, contract risk factors, Customer 360).


In [ ]:
import sys
from pathlib import Path

# Setup project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from databricks.bronze_ingestion import BronzeIngestionPipeline
from databricks.silver_cleaning import SilverCleaningPipeline
from databricks.gold_feature_store import GoldFeatureStorePipeline
from databricks.pipeline_orchestrator import MedallionPipelineOrchestrator

print("Databricks Medallion Architecture modules loaded successfully!")


### 1. Execute Bronze Ingestion Tier


In [ ]:
bronze = BronzeIngestionPipeline()
bronze_df = bronze.ingest_customers_raw()
print(f"Bronze Delta Table Record Count: {len(bronze_df)}")
bronze_df[["customerID", "gender", "tenure", "MonthlyCharges", "_bronze_ingested_at", "_bronze_batch_id"]].head()


### 2. Execute Silver Cleansing & Validation Tier


In [ ]:
silver = SilverCleaningPipeline()
silver_df, quality_report = silver.clean_customers(bronze_df)
print("Silver Quality Report:", quality_report)
silver_df[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn_Numeric", "_silver_processed_at"]].head()


### 3. Execute Gold Feature Store & Aggregation Tier


In [ ]:
gold = GoldFeatureStorePipeline()
gold_df = gold.build_gold_features(silver_df)
print(f"Gold Features Shape: {gold_df.shape}")
gold_df[["customerID", "tenure_cohort", "contract_risk_index", "security_bundle_score", "total_active_services", "estimated_clv"]].head()


### 4. End-to-End Orchestration Summary


In [ ]:
orchestrator = MedallionPipelineOrchestrator()
summary = orchestrator.run_pipeline()
print("Pipeline Status:", summary["status"])
print(f"Total Runtime: {summary['total_runtime_seconds']}s")
